# DNS / GP Macro Panel Builder (1972–Present)

This notebook downloads the macro-financial indicator set for the DNS / Macro-DNS / GP workflow and writes a **single merged CSV** from **1972 to present**.

It uses the corrected Diebold-Li 2006 replication convention from the prior notebook for the original monthly benchmark variables:

- `FEDFUNDS -> FFR`
- `CUMFNS -> CU`
- `PCEPI -> PI` via 12-month inflation

For those **monthly** benchmark-style variables, the month label is shifted **back by one month**, matching the corrected alignment from the previous notebook.

Additional indicators are added for the broader GP state vector:

- Effective federal funds rate (`DFF`) converted to a monthly last-business-day snapshot
- 10Y–2Y Treasury spread (`T10Y2Y`) converted to a monthly last-business-day snapshot
- Moody's Baa yield (`BAA`)
- Moody's Aaa yield (`AAA`)
- Major-currency dollar index (spliced from `DTWEXM` and `TWEXAFEGSMTH`)
- Global 10-year sovereign yield principal component from:
  - Canada `IRLTLT01CAM156N`
  - Germany `IRLTLT01DEM156N`
  - United Kingdom `IRLTLT01GBM156N`
  - France `IRLTLT01FRM156N`

The final output is written to:
- `dns_gp_macro_panel_1972_present.csv`


In [1]:

# =========================
# 1. Imports and settings
# =========================
import os
import time
import requests
import warnings
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

API_KEY = "8646e4c441bf17d906a058f8eddaf3f8"

if not API_KEY:
    raise ValueError("Set FRED_API_KEY in your environment or assign API_KEY directly in this cell.")

FRED_OBS_URL = "https://api.stlouisfed.org/fred/series/observations"

# Start one year earlier so 12-month inflation is available beginning in 1972
HISTORY_START = "1971-01-01"
PANEL_START_MONTH = pd.Period("1972-01", freq="M")
PANEL_END_DATE = pd.Timestamp.today().strftime("%Y-%m-%d")

OUTDIR = Path("./dns_gp_macro_output")
OUTDIR.mkdir(exist_ok=True)

MONTHLY_SERIES = {
    "FEDFUNDS": "Federal Funds Rate",
    "CUMFNS": "Capacity Utilization: Manufacturing",
    "PCEPI": "PCE Price Index",
    "BAA": "Moody's Seasoned Baa Corporate Bond Yield",
    "AAA": "Moody's Seasoned Aaa Corporate Bond Yield",
    "DTWEXM": "Major-Currency Dollar Index (discontinued)",
    "TWEXAFEGSMTH": "Nominal Advanced Foreign Economies Dollar Index",
    "IRLTLT01CAM156N": "Canada 10Y government bond yield",
    "IRLTLT01DEM156N": "Germany 10Y government bond yield",
    "IRLTLT01GBM156N": "United Kingdom 10Y government bond yield",
    "IRLTLT01FRM156N": "France 10Y government bond yield",
}

DAILY_SERIES = {
    "DFF": "Effective Federal Funds Rate",
    "T10Y2Y": "10Y minus 2Y Treasury spread",
}


In [2]:

# =========================
# 2. API helpers
# =========================
def _request_fred(url, params, max_retries=5, pause=0.75, verbose=False):
    last_response = None
    for attempt in range(max_retries):
        r = requests.get(url, params=params, timeout=60)
        last_response = r
        if r.status_code == 200:
            return r
        if verbose:
            print(f"Attempt {attempt+1} failed with status {r.status_code}")
            try:
                print(r.json())
            except Exception:
                print(r.text[:1000])
        time.sleep(pause * (attempt + 1))
    last_response.raise_for_status()

def fetch_fred_series(series_id, observation_start=HISTORY_START, observation_end=PANEL_END_DATE):
    params = {
        "series_id": series_id,
        "api_key": API_KEY,
        "file_type": "json",
        "observation_start": observation_start,
        "observation_end": observation_end,
        "sort_order": "asc",
    }
    r = _request_fred(FRED_OBS_URL, params)
    obs = r.json()["observations"]
    df = pd.DataFrame(obs)

    if df.empty:
        return pd.DataFrame(columns=["date", series_id])

    df["date"] = pd.to_datetime(df["date"], errors="coerce")
    df[series_id] = pd.to_numeric(df["value"], errors="coerce")
    return df[["date", series_id]].sort_values("date").reset_index(drop=True)

def last_business_day_monthly(df, value_col):
    out = df.copy()
    out = out.dropna(subset=["date"])
    out["month"] = out["date"].dt.to_period("M")
    out = out.sort_values("date").groupby("month", as_index=False).tail(1).copy()
    out = out.rename(columns={"date": f"{value_col}_obs_date"})
    return out[["month", f"{value_col}_obs_date", value_col]].reset_index(drop=True)

def corrected_month_label_monthly(df, value_cols):
    '''
    Apply the same benchmark-style month-label correction used in the prior notebook:
    a row dated YYYY-MM-01 is labeled to the PREVIOUS month.
    '''
    out = df.copy()
    out["month"] = (out["date"] - pd.offsets.MonthBegin(1)).dt.to_period("M")
    keep = ["date", "month"] + value_cols
    return out[keep].copy()

def make_month_string(period_series):
    return period_series.astype(str)


In [3]:

# =========================
# 3. Download monthly and daily source series
# =========================
monthly_histories = {}
for sid, desc in MONTHLY_SERIES.items():
    print(f"Downloading monthly series: {sid} | {desc}")
    monthly_histories[sid] = fetch_fred_series(sid)

daily_histories = {}
for sid, desc in DAILY_SERIES.items():
    print(f"Downloading daily series: {sid} | {desc}")
    daily_histories[sid] = fetch_fred_series(sid)

print("Done.")


Done.


In [4]:

# =========================
# 4. Build corrected monthly DNS benchmark block
# =========================
ffr = monthly_histories["FEDFUNDS"].copy()
cu = monthly_histories["CUMFNS"].copy()
pcepi = monthly_histories["PCEPI"].copy()

macro_core = (
    ffr.merge(cu, on="date", how="outer")
       .merge(pcepi, on="date", how="outer")
       .sort_values("date")
       .reset_index(drop=True)
)

# Inflation on the source monthly timeline
macro_core["PI"] = 100.0 * (macro_core["PCEPI"] / macro_core["PCEPI"].shift(12) - 1.0)

macro_core = macro_core.rename(columns={
    "FEDFUNDS": "FFR",
    "CUMFNS": "CU"
})

# Correct benchmark month label back by one month
macro_core = corrected_month_label_monthly(macro_core, ["FFR", "CU", "PCEPI", "PI"])

macro_core = macro_core[
    macro_core["month"] >= PANEL_START_MONTH
].reset_index(drop=True)

macro_core.head()


,date,month,FFR,CU,PCEPI,PI
0,1972-02-01,1972-01,3.30,81.8563,21.098,3.951518
1,1972-03-01,1972-02,3.83,82.2062,21.128,3.756814
2,1972-04-01,1972-03,4.17,82.8926,21.160,3.507313
3,1972-05-01,1972-04,4.27,82.8225,21.207,3.307677
4,1972-06-01,1972-05,4.46,82.8930,21.239,3.006935


In [5]:

# =========================
# 5. Build additional monthly GP indicators
# =========================
baa = corrected_month_label_monthly(monthly_histories["BAA"].copy(), ["BAA"])
aaa = corrected_month_label_monthly(monthly_histories["AAA"].copy(), ["AAA"])
dtwexm = corrected_month_label_monthly(monthly_histories["DTWEXM"].copy(), ["DTWEXM"])
twex = corrected_month_label_monthly(monthly_histories["TWEXAFEGSMTH"].copy(), ["TWEXAFEGSMTH"])

can10 = corrected_month_label_monthly(monthly_histories["IRLTLT01CAM156N"].copy(), ["IRLTLT01CAM156N"])
deu10 = corrected_month_label_monthly(monthly_histories["IRLTLT01DEM156N"].copy(), ["IRLTLT01DEM156N"])
gbr10 = corrected_month_label_monthly(monthly_histories["IRLTLT01GBM156N"].copy(), ["IRLTLT01GBM156N"])
fra10 = corrected_month_label_monthly(monthly_histories["IRLTLT01FRM156N"].copy(), ["IRLTLT01FRM156N"])

for _df in [baa, aaa, dtwexm, twex, can10, deu10, gbr10, fra10]:
    _df.drop(columns=["date"], inplace=True)
    _df = _df[_df["month"] >= PANEL_START_MONTH]


In [6]:

# =========================
# 6. Splice the major-currency dollar index
# =========================
dollar_overlap = dtwexm.merge(twex, on="month", how="inner")

if dollar_overlap.empty:
    raise ValueError("No overlap between DTWEXM and TWEXAFEGSMTH for dollar index splice.")

scale = dollar_overlap["DTWEXM"].mean() / dollar_overlap["TWEXAFEGSMTH"].mean()

twex["major_dollar"] = twex["TWEXAFEGSMTH"] * scale
dtwexm["major_dollar"] = dtwexm["DTWEXM"]

major_dollar = pd.concat([
    dtwexm[["month", "major_dollar"]],
    twex.loc[~twex["month"].isin(dtwexm["month"]), ["month", "major_dollar"]]
], ignore_index=True).sort_values("month").drop_duplicates("month", keep="first").reset_index(drop=True)

major_dollar = major_dollar[major_dollar["month"] >= PANEL_START_MONTH].copy()
major_dollar.head()


,month,major_dollar
0,1973-01,108.2242
1,1973-02,101.4009
2,1973-03,100.4158
3,1973-04,100.7760
4,1973-05,98.8659


In [7]:

# =========================
# 7. Build global 10Y sovereign yield PC1
# =========================
global_10y = (
    can10[["month", "IRLTLT01CAM156N"]]
      .merge(deu10[["month", "IRLTLT01DEM156N"]], on="month", how="outer")
      .merge(gbr10[["month", "IRLTLT01GBM156N"]], on="month", how="outer")
      .merge(fra10[["month", "IRLTLT01FRM156N"]], on="month", how="outer")
      .rename(columns={
          "IRLTLT01CAM156N": "can10",
          "IRLTLT01DEM156N": "deu10",
          "IRLTLT01GBM156N": "gbr10",
          "IRLTLT01FRM156N": "fra10",
      })
      .sort_values("month")
      .reset_index(drop=True)
)

# Conservative fill only inside the sample to avoid dropping the entire early window
global_10y[["can10", "deu10", "gbr10", "fra10"]] = global_10y[["can10", "deu10", "gbr10", "fra10"]].ffill().bfill()

pc_input = global_10y[["can10", "deu10", "gbr10", "fra10"]].dropna()
if pc_input.empty:
    global_10y["global10y_pc1"] = np.nan
else:
    scaler = StandardScaler()
    X_std = scaler.fit_transform(pc_input.values)
    pca = PCA(n_components=1)
    pc1 = pca.fit_transform(X_std)[:, 0]
    global_10y.loc[pc_input.index, "global10y_pc1"] = pc1

global_10y = global_10y[global_10y["month"] >= PANEL_START_MONTH].copy()
global_10y.head()


,month,can10,deu10,gbr10,fra10,global10y_pc1
13,1972-01,6.87,7.3,7.05,8.33,0.689561
14,1972-02,7.07,7.4,7.39,8.07,0.743542
15,1972-03,7.32,7.7,7.44,7.92,0.815150
16,1972-04,7.33,7.8,7.85,7.90,0.879636
17,1972-05,7.39,7.9,8.95,7.83,1.027848


In [8]:

# =========================
# 8. Build daily snapshot block (month-end market values)
# =========================
dff_monthly = last_business_day_monthly(daily_histories["DFF"], "DFF")
t10y2y_monthly = last_business_day_monthly(daily_histories["T10Y2Y"], "T10Y2Y")

dff_monthly = dff_monthly[dff_monthly["month"] >= PANEL_START_MONTH].copy()
t10y2y_monthly = t10y2y_monthly[t10y2y_monthly["month"] >= PANEL_START_MONTH].copy()

dff_monthly.head()


,month,DFF_obs_date,DFF
12,1972-01,1972-01-31,3.13
13,1972-02,1972-02-29,3.25
14,1972-03,1972-03-31,4.00
15,1972-04,1972-04-30,4.31
16,1972-05,1972-05-31,4.75


In [9]:

# =========================
# 9. Merge everything into a single panel
# =========================
panel = macro_core[["month", "FFR", "CU", "PCEPI", "PI"]].copy()

panel = (
    panel.merge(baa[["month", "BAA"]], on="month", how="left")
         .merge(aaa[["month", "AAA"]], on="month", how="left")
         .merge(dff_monthly[["month", "DFF"]], on="month", how="left")
         .merge(t10y2y_monthly[["month", "T10Y2Y"]], on="month", how="left")
         .merge(major_dollar, on="month", how="left")
         .merge(global_10y[["month", "can10", "deu10", "gbr10", "fra10", "global10y_pc1"]], on="month", how="left")
         .sort_values("month")
         .reset_index(drop=True)
)

panel["month_str"] = make_month_string(panel["month"])
panel["date_month_end"] = panel["month"].dt.to_timestamp("M")
panel["date_month_start"] = panel["month"].dt.to_timestamp("D")

# Reorder columns
panel = panel[[
    "month", "month_str", "date_month_start", "date_month_end",
    "FFR", "CU", "PCEPI", "PI",
    "DFF", "T10Y2Y", "BAA", "AAA", "major_dollar",
    "can10", "deu10", "gbr10", "fra10", "global10y_pc1"
]]

panel.head(15)


,month,month_str,date_month_start,date_month_end,FFR,CU,PCEPI,PI,DFF,T10Y2Y,BAA,AAA,major_dollar,can10,deu10,gbr10,fra10,global10y_pc1
0,1972-01,1972-01,1972-01-01,1972-01-31,3.30,81.8563,21.098,3.951518,3.13,NaN,8.23,7.27,NaN,6.87,7.3,7.05,8.33,0.689561
1,1972-02,1972-02,1972-02-01,1972-02-29,3.83,82.2062,21.128,3.756814,3.25,NaN,8.24,7.24,NaN,7.07,7.4,7.39,8.07,0.743542
2,1972-03,1972-03,1972-03-01,1972-03-31,4.17,82.8926,21.160,3.507313,4.00,NaN,8.24,7.30,NaN,7.32,7.7,7.44,7.92,0.815150
3,1972-04,1972-04,1972-04-01,1972-04-30,4.27,82.8225,21.207,3.307677,4.31,NaN,8.23,7.30,NaN,7.33,7.8,7.85,7.90,0.879636
4,1972-05,1972-05,1972-05-01,1972-05-31,4.46,82.8930,21.239,3.006935,4.75,NaN,8.20,7.23,NaN,7.39,7.9,8.95,7.83,1.027848
5,1972-06,1972-06,1972-06-01,1972-06-30,4.55,82.6441,21.315,3.065616,4.50,NaN,8.23,7.21,NaN,7.48,8.0,8.90,7.80,1.046990
6,1972-07,1972-07,1972-07-01,1972-07-31,4.81,83.5285,21.377,3.066390,4.50,NaN,8.19,7.19,NaN,7.46,7.9,9.05,7.86,1.052979
7,1972-08,1972-08,1972-08-01,1972-08-31,4.87,83.9714,21.473,3.374735,5.13,NaN,8.09,7.22,NaN,7.48,7.9,9.29,7.83,1.080978
8,1972-09,1972-09,1972-09-01,1972-09-30,5.05,84.9134,21.497,3.331090,4.88,NaN,8.06,7.21,NaN,7.34,8.0,9.10,7.91,1.064671
9,1972-10,1972-10,1972-10-01,1972-10-31,5.06,85.7175,21.561,3.424953,4.88,NaN,7.99,7.12,NaN,7.18,8.4,9.09,8.02,1.119631


In [10]:

# =========================
# 10. Save outputs
# =========================
panel_to_save = panel.copy()
panel_to_save["month"] = panel_to_save["month"].astype(str)

final_csv = OUTDIR / "dns_gp_macro_panel_1972_present.csv"
panel_to_save.to_csv(final_csv, index=False)

print(f"Saved: {final_csv}")
print(f"Rows: {len(panel_to_save):,}")
print(f"Month range: {panel_to_save['month'].min()} to {panel_to_save['month'].max()}")

display(panel.tail(15))


Saved: dns_gp_macro_output/dns_gp_macro_panel_1972_present.csv
Rows: 650
Month range: 1972-01 to 2026-02


,month,month_str,date_month_start,date_month_end,FFR,CU,PCEPI,PI,DFF,T10Y2Y,BAA,AAA,major_dollar,can10,deu10,gbr10,fra10,global10y_pc1
635,2024-12,2024-12,2024-12-01,2024-12-31,4.33,74.6327,125.417,2.607380,4.33,0.33,6.08,5.46,117.273244,3.288636,2.483636,4.6627,3.32,-1.462623
636,2025-01,2025-01,2025-01-01,2025-01-31,4.33,75.5230,125.921,2.710485,4.33,0.36,5.92,5.32,116.102875,3.056316,2.405500,4.5063,3.15,-1.546006
637,2025-02,2025-02,2025-02-01,2025-02-28,4.33,75.7850,125.941,2.359434,4.33,0.25,5.93,5.29,113.574325,3.010476,2.741429,4.6448,3.43,-1.448022
638,2025-03,2025-03,2025-03-01,2025-03-31,4.33,75.6285,126.150,2.277426,4.33,0.34,6.18,5.45,109.968393,3.108571,2.510500,4.5762,3.26,-1.500401
639,2025-04,2025-04,2025-04-01,2025-04-30,4.33,75.4680,126.380,2.458086,4.33,0.57,6.29,5.54,109.296021,3.221429,2.562857,4.6004,3.26,-1.473463
640,2025-05,2025-05,2025-05-01,2025-05-31,4.33,75.6385,126.743,2.593513,4.33,0.52,6.15,5.46,107.433753,3.314000,2.519048,4.5248,3.24,-1.479305
641,2025-06,2025-06,2025-06-01,2025-06-30,4.33,75.9194,126.960,2.605547,4.33,0.52,6.10,5.45,107.104430,3.480455,2.631304,4.5924,3.36,-1.415931
642,2025-07,2025-07,2025-07-01,2025-07-31,4.33,75.8608,127.293,2.747621,4.33,0.43,6.00,5.35,107.529641,3.422500,2.673333,4.6369,3.42,-1.404652
643,2025-08,2025-08,2025-08-01,2025-08-31,4.22,75.7980,127.625,2.787442,4.33,0.64,5.83,5.21,107.078235,3.224500,2.693182,4.6885,3.51,-1.411847
644,2025-09,2025-09,2025-09-01,2025-09-30,4.09,75.1411,127.871,2.712581,4.09,0.56,5.74,5.13,108.205300,3.128182,2.617826,4.5721,3.44,-1.459546
